# Fraud-risk baseline

> **Dữ liệu synthetic:** kết quả chỉ phục vụ kiểm tra pipeline, không đại diện cho khách hàng thực.

Notebook này dùng lại hàm trong `src/`; đưa repository và `data/features/` lên Kaggle, rồi cập nhật `PROJECT_ROOT`. Các metric gồm RMSE, MAE, R², Spearman và precision/recall tại top-k trên validation.

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    raise FileNotFoundError("Không tìm thấy configs/config.yaml từ working directory hiện tại.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.train_fraud import train_fraud_baselines
from src.utils.io import read_parquet_safe

In [2]:
with (PROJECT_ROOT / 'configs/model_fraud.yaml').open(encoding='utf-8') as handle:
    fraud_config = yaml.safe_load(handle)

feature_dir = PROJECT_ROOT / 'data/features'
X_train = read_parquet_safe(feature_dir / 'X_train.parquet')
X_valid = read_parquet_safe(feature_dir / 'X_valid.parquet')
y_train = read_parquet_safe(feature_dir / 'y_fraud_train.parquet')['fraud_risk_score']
y_valid = read_parquet_safe(feature_dir / 'y_fraud_valid.parquet')['fraud_risk_score']

{"timestamp": "2026-11-04T02:34:33.712568+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\X_train.parquet"}
{"timestamp": "2026-11-04T02:34:34.005599+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\X_valid.parquet"}
{"timestamp": "2026-11-04T02:34:34.031434+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\y_fraud_train.parquet"}
{"timestamp": "2026-11-04T02:34:34.054536+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\y_fraud_valid.parquet"}


In [3]:
# Set log_mlflow=True when you want local Kaggle MLflow records under mlruns/.
fraud_results = train_fraud_baselines(
    X_train, y_train, X_valid, y_valid, fraud_config, seed=42, log_mlflow=False
)
pd.DataFrame({name: result.metrics for name, result in fraud_results.items()}).T

{"timestamp": "2026-11-04T02:34:37.490814+00:00", "level": "INFO", "logger": "src.utils.report", "message": "Wrote HTML report: reports\\fraud_baseline_dummy.html"}
{"timestamp": "2026-11-04T02:34:37.491712+00:00", "level": "INFO", "logger": "src.models.train_fraud", "message": "Fraud baseline dummy validation metrics: {'rmse': 0.15941378247536506, 'mae': 0.12159444944592596, 'r2': -1.211312848470314e-06, 'spearman': nan, 'precision_at_k': 0.04515555555555555, 'recall_at_k': 0.04515555555555555, 'k': 0.05, 'top_k_count': 5625.0}"}
{"timestamp": "2026-11-04T02:34:38.212662+00:00", "level": "INFO", "logger": "src.utils.report", "message": "Wrote HTML report: reports\\fraud_baseline_ridge.html"}
{"timestamp": "2026-11-04T02:34:38.213530+00:00", "level": "INFO", "logger": "src.models.train_fraud", "message": "Fraud baseline ridge validation metrics: {'rmse': 0.112264658471691, 'mae': 0.08554325786422685, 'r2': 0.5040534061364341, 'spearman': 0.6954532628136015, 'precision_at_k': 0.4624, 'r

,rmse,mae,r2,spearman,precision_at_k,recall_at_k,k,top_k_count
dummy,0.159414,0.121594,-0.000001,NaN,0.045156,0.045156,0.05,5625.0
ridge,0.112265,0.085543,0.504053,0.695453,0.462400,0.462400,0.05,5625.0
